## Silver Layer — Flights

Cleans the raw Aviationstack data in `bronze_flight` and saves it to `silver_flights`,
which `03_transform_gold` uses to build the flight, airline and airport tables. Each run
also adds the flights to `silver_flights_history`, which feeds the delay insights.

```text
workspace.default.bronze_flight  →  workspace.default.silver_flights  →  silver_flights_history
```

### What gets cleaned

| Step | Raw data problem | What Silver does |
|---|---|---|
| **1. Codeshares Aviationstack flags** | One plane is sold under several airlines' flight numbers, and each number is its own row. The copies name the operating flight in `codeshared_*` | Uses the operating flight's number and airline on every copy |
| **2. Codeshares it doesn't flag** | `codeshared` is often empty, so copies look like separate flights | Treats rows with the same date, route, scheduled departure, gate and terminal as one physical flight. Keeps the operating airline's row, otherwise the lowest flight number (codeshare numbers are usually the high ones). `codeshare_count` records how many copies were merged |
| **3. Lower-case codes** | Codeshare fields arrive in lower case, e.g. `tg960` | Upper-cases airline and flight codes |
| **4. Airline names** | Codeshare rows name the airline in lower case, e.g. `air france` | Takes the proper name from the airline's own rows, otherwise capitalises it |
| **5. Stray whitespace** | Names can carry spaces and line breaks | Strips them from airline and airport names |
| **6. No flight number** | A few rows have none, so they can't be identified | Drops them |
| **7. Times as text** | Times are strings, e.g. `departure_scheduled` | Converts them to timestamps and renames them, e.g. `scheduled_departure` |
| **8. Refetched flights** | A rerun on the same day returns the same flights again | The history MERGE updates them instead of adding them twice |

### Not cleaned yet

- **Local times labelled UTC:** each airport's local time arrives marked `+00:00` and is kept
  as-is. Gold reads it in UTC so the clock time stays right; converting to true UTC here, using
  `departure_timezone`, would be cleaner.
- **Delays aren't checked:** `departure_delay` isn't compared with actual minus scheduled time.
  Gold recalculates the delay from the times instead.
- **Rare false merges:** two genuinely different flights with the same route and departure minute,
  and no gate or terminal, would be counted as one.


In [ ]:
from pyspark.sql import Window
from pyspark.sql.functions import *

bronze_df = spark.table("workspace.default.bronze_flight")

is_codeshare = col("codeshared_flight_iata").isNotNull()


def clean(name):
    """Strip surrounding whitespace, including line breaks (Spark's trim only removes spaces)."""
    return regexp_replace(name, r"^\s+|\s+$", "")

# Codeshare rows give the operating airline in lower case (e.g. "air france", "tg960"), so codes
# are upper-cased below, and the proper airline name is taken
# from an operating row of the same airline where there is one.
airline_names = bronze_df \
    .filter(col("codeshared_flight_iata").isNull() & col("airline_iata").isNotNull()) \
    .groupBy(col("airline_iata").alias("operating_iata")) \
    .agg(first("airline_name", ignorenulls=True).alias("operating_name"))

operating = bronze_df \
    .join(airline_names, upper(bronze_df.codeshared_airline_iata) == airline_names.operating_iata, "left") \
    .select(
        "*",
        is_codeshare.alias("is_codeshare"),
        when(is_codeshare, coalesce(col("operating_name"), initcap("codeshared_airline_name")))
            .otherwise(col("airline_name")).alias("op_airline_name"),
        upper(when(is_codeshare, col("codeshared_airline_iata")).otherwise(col("airline_iata"))).alias("op_airline_iata"),
        upper(when(is_codeshare, col("codeshared_airline_icao")).otherwise(col("airline_icao"))).alias("op_airline_icao"),
        when(is_codeshare, col("codeshared_flight_number")).otherwise(col("flight_number")).alias("op_flight_number"),
        upper(when(is_codeshare, col("codeshared_flight_iata")).otherwise(col("flight_iata"))).alias("op_flight_iata"),
        upper(when(is_codeshare, col("codeshared_flight_icao")).otherwise(col("flight_icao"))).alias("op_flight_icao")
    )

# One row per physical flight. Aviationstack often leaves `codeshared` empty, so copies are
# also recognised by what only one plane can share: route, departure minute and gate.
# The operating airline's own row is kept when known; otherwise the lowest flight number,
# since codeshare numbers are usually the high ones (e.g. 8xxx).
physical_flight = Window.partitionBy(
    "flight_date", "departure_iata", "arrival_iata", "departure_scheduled",
    coalesce(col("departure_gate"), lit("")), coalesce(col("departure_terminal"), lit(""))
)
operating_first = physical_flight.orderBy(
    col("is_codeshare").asc(),
    col("op_flight_number").cast("int").asc_nulls_last(),
    col("op_flight_iata").asc()
)

silver_flights = operating \
    .filter(col("op_flight_iata").isNotNull()) \
    .withColumn("row", row_number().over(operating_first)) \
    .withColumn("codeshare_count", count(lit(1)).over(physical_flight) - 1) \
    .filter(col("row") == 1) \
    .select(
        "flight_date",
        "flight_status",
        col("op_flight_iata").alias("flight_iata"),
        col("op_flight_icao").alias("flight_icao"),
        col("op_flight_number").alias("flight_number"),
        clean(col("op_airline_name")).alias("airline_name"),
        col("op_airline_iata").alias("airline_iata"),
        col("op_airline_icao").alias("airline_icao"),
        clean(col("departure_airport")).alias("departure_airport"),
        "departure_timezone",
        "departure_iata",
        "departure_icao",
        "departure_terminal",
        "departure_gate",
        clean(col("arrival_airport")).alias("arrival_airport"),
        "arrival_timezone",
        "arrival_iata",
        "arrival_icao",
        "arrival_terminal",
        "arrival_baggage",
        "departure_delay",
        "arrival_delay",
        to_timestamp("departure_scheduled").alias("scheduled_departure"),
        to_timestamp("departure_estimated").alias("estimated_departure"),
        to_timestamp("departure_actual").alias("actual_departure"),
        to_timestamp("arrival_scheduled").alias("scheduled_arrival"),
        to_timestamp("arrival_estimated").alias("estimated_arrival"),
        to_timestamp("arrival_actual").alias("actual_arrival"),
        "codeshare_count"
    )

print("bronze rows:", bronze_df.count())
print("rows marked as codeshare by Aviationstack:", bronze_df.filter(col("codeshared_flight_iata").isNotNull()).count())
print("silver flights:", silver_flights.count())
display(silver_flights.limit(10))

### Save to Silver

Replaces `silver_flights` with the latest cleaned flights. `overwriteSchema` lets the
table pick up new columns, such as `codeshare_count`.

In [ ]:
silver_flights.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.silver_flights")

saved = spark.table("workspace.default.silver_flights")
print("silver_flights rows:", saved.count())
print("delayed flights:", saved.filter((col("departure_delay") > 0) | (col("arrival_delay") > 0)).count())

### History

`silver_flights` only holds the latest fetch. Each run also adds its flights to
`silver_flights_history`, which keeps every day and feeds the delay insight tables in Gold.

It uses a Delta **MERGE** keyed on flight date and flight number: a flight fetched again on
the same day is **updated** rather than added twice, and new flights are inserted.

In [ ]:
from delta.tables import DeltaTable

HISTORY = "workspace.default.silver_flights_history"

latest = spark.table("workspace.default.silver_flights").withColumn("ingested_at", current_timestamp())

if not spark.catalog.tableExists(HISTORY):
    latest.write.format("delta").saveAsTable(HISTORY)
else:
    DeltaTable.forName(spark, HISTORY).alias("history") \
        .merge(latest.alias("latest"), "history.flight_date = latest.flight_date AND history.flight_iata = latest.flight_iata") \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()

history = spark.table(HISTORY)
print("history rows:", history.count())
display(
    history.groupBy("departure_iata")
    .agg(count("*").alias("flights"), countDistinct("flight_date").alias("days"), min("flight_date").alias("first_day"), max("flight_date").alias("last_day"))
    .orderBy(desc("flights"))
)